# 02 理论证明审计（Tier A）

论文的两个核心引理都不是对任意网络的严格定理，而是建立在一个**替代模型**上的性质：

- **Lemma 1（脏标签）**：在 NTK 核 $K(x,x')=e^{-2\gamma\lVert x-x'\rVert^2}$ 下，
  目标类别的核和超过其他类别，因此任务无关图像会被推向目标类。
- **Lemma 2（损坏）**：对图像做损坏后，目标标签的概率下降、真值标签的概率上升。

本 Notebook 做的三件事：

1. 逐元素检查 Lemma 2 证明中的代数步骤（Eq.29 ↔ Eq.30）；
2. 在真实 CIFAR-10 像素上按论文的核函数计算核和，检查 Lemma 1 的两种情形；
3. 检查干净标签分支 Eq.35 的一次不等式放缩是否对所有取值成立。

> **注意**：这些检查复现的是“论文证明中的计算”，不是“对训练好的深度网络的证明”。
> 因此即使数值支持，也不能把 Lemma 1 升级为 CONFIRMED。


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


## A09：Lemma 1（脏标签）的核和占优

论文证明中的核函数（Eq.13）为

$$K_\gamma(x,x')=\exp\left(-2\gamma\lVert x-x'\rVert^2\right).$$

对查询图 $x$（伪标签 $k_1\neq$ 目标类 $t$），论文要求目标类的核和大于其他任一非 $k_1$ 类：

$$\sum_{j:\,y_j=t}K(x,x_j) > \max_{k\notin\{k_1,t\}}\sum_{j:\,y_j=k}K(x,x_j).$$

代码用灰度像素（32×32，归一化到 [0,1]），$\gamma$ 由查询-训练距离中位数标定，
并在 $0.5\times,1\times,2\times$ 下检查稳健性。两个情形：

- `case1_nt_gt_nk2`：目标类 100 干净 + 90 中毒 = 190 张，其他类各 100 张；
- `case2_nt_eq_nk2`：目标类 80 干净 + 20 中毒 = 100 张，与其他类数量相同。

若最小占优比例 ≥ 0.5 记 PARTIAL（数值支持），否则 CONTRADICTED。数值模拟永远不能判 CONFIRMED。


In [ ]:
r = nbutil.show("A09")


## A10：干净标签分支与 Eq.35 的放缩

干净标签下目标类没有“中毒图”，证明需要处理 $v=x+T-x_t$。Eq.35 声称

$$e^{-2\gamma\lVert v-T\rVert^2}\ \ge\ e^{-2\gamma(\lVert v\rVert^2+\lVert T\rVert^2)}.$$

但这只在 $\langle v,T\rangle\ge 0$ 时成立；证明中没有声明该条件。
取 $a=0,t=1,b=1.5,n=2$，则 $v=-0.5$：

- 左边 $=e^{-2(a-b)^2}=e^{-4.5}\approx0.0111$；
- 右边 $=e^{-2((u-b)^2+t^2)}=e^{-2.5}\approx0.0821$。

方向相反，放缩无效。代码同时报告该反例与干净标签协议下的核和模拟。


In [ ]:
r = nbutil.show("A10")
print("\n反例：", json.dumps(r.measured["counterexample"], ensure_ascii=False))


## A07：Eq.29 ↔ Eq.30 的代数等价性

Lemma 2 的证明需要把

$$\left(\frac{a+t}{n}-b\right)^2>(a+t-b)^2 \tag{Eq.29}$$

化为

$$t<\frac{2nb}{1+n}-a. \tag{Eq.30}$$

两边开方时默认了 $a+t>0$（像素空间的合理取值），这一步论文没有写。
在 $a+t>0$ 时等价成立；在 $a+t<0$ 时方向翻转。下面各取 4000 组随机数验证。


In [ ]:
r = nbutil.show("A07")
print("u>0 反例数:", r.measured["mismatches_positive_u"])
print("u<0 反例数:", r.measured["mismatches_negative_u"])
print("系数 2n/(1+n):", [round(x, 4) for x in r.measured["coefficient_2n_over_1_plus_n"]])


## A08：Lemma 2 是“逐损坏普遍成立”还是“聚合趋势”？

论文 Lemma 2 的表述是逐图、逐损坏的：对任意损坏 $j$，目标标签概率下降且真值标签概率上升。
但正文引用它时通常只说“聚合超过 90%”。这两个强度不同的断言必须分开检查。

代码在缩比 BadNets 模型上取 100 张非目标测试图，施加全部 80 种损坏，统计同时满足两条的
`(图, 损坏)` 比例。判定规则：

- 普遍比例 = 1.0 也只能记 PARTIAL（单个代理模型不能证明普适命题）；
- 聚合比例 ≥ 0.90 → PARTIAL；
- 否则 → CONTRADICTED。


In [ ]:
r = nbutil.show("A08", max_measured_chars=2500)
print("普遍比例:", round(r.measured.get("universal_fraction", -1), 4))
print("聚合比例:", round(r.measured.get("aggregate_fraction", -1), 4))


## 本 Notebook 的小结

| 声明 | 结论 | 要点 |
|---|---|---|
| A07 | PARTIAL | Eq.29↔Eq.30 在 $a+t>0$ 时成立，但证明未声明该前提 |
| A08 | PARTIAL/CONTRADICTED | 逐图逐损坏的普遍形式通常被反例否定；聚合趋势另算 |
| A09 | PARTIAL/CONTRADICTED | NTK 核和是否占优取决于 γ 与样本构成 |
| A10 | CONTRADICTED | Eq.35 的放缩缺少 $\langle v,T\rangle\ge0$，存在明确反例 |

引理是论文动机的一部分，不是方法能否工作的充要条件；方法本身的检验在 `04_detection` 与 `05_removal`。
